# G1 Academy Bonus - Task 10: end-effector IK control + teleoperation UI

## Introduction
This task builds a self-contained, position-only `ik_move_ee(hand, dx, dy, dz)` step function from the G1 URDF arm geometry, then wraps it in a small Jupyter UI to teleoperate the end effector - jog buttons for +/-X/Y/Z, plus FSM, arm handoff, and hand controls.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

import sys
if ".." not in sys.path:
    sys.path.append("..")
from sdk_wrapper import ensure_channel_factory

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Reuse Task 8's upper-body pose/`arm_sdk` helpers, and Dex3 basics

In [ ]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_, unitree_hg_msg_dds__HandCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, HandCmd_
from unitree_sdk2py.utils.crc import CRC

WAIST_JOINTS = (12, 13, 14)
UPPER_BODY_JOINTS = list(WAIST_JOINTS) + list(range(15, 22)) + list(range(22, 29))
LEFT_ARM_JOINTS = list(range(15, 22)); RIGHT_ARM_JOINTS = list(range(22, 29))
_crc = CRC()
arm_sdk_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); arm_sdk_pub.Init()

def current_upper_body_pose(timeout_s=3.0):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if lowstate_sub.message is not None:
            return {j: float(lowstate_sub.message.motor_state[j].q) for j in UPPER_BODY_JOINTS}
        time.sleep(0.02)
    raise TimeoutError("No fresh rt/lowstate.")

def write_arm_sdk_pose(targets, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0):
    msg = unitree_hg_msg_dds__LowCmd_(); msg.mode_pr = 0; msg.mode_machine = 0
    msg.motor_cmd[29].q = max(0.0, min(1.0, float(weight)))
    for joint, q in targets.items():
        cmd = msg.motor_cmd[int(joint)]
        cmd.mode = 1; cmd.q = float(q); cmd.dq = 0.0; cmd.tau = 0.0
        cmd.kp = waist_kp if int(joint) in WAIST_JOINTS else kp
        cmd.kd = waist_kd if int(joint) in WAIST_JOINTS else kd
    msg.crc = _crc.Crc(msg)
    arm_sdk_pub.Write(msg)

def release_arms(steps=150, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        ratio = i / steps; fade = ratio * ratio * (3 - 2 * ratio); weight = 1.0 - fade
        write_arm_sdk_pose(pose, weight=weight, kp=30.0 * weight, kd=1.5 * weight, waist_kp=480.0 * weight, waist_kd=12.0 * weight)
        time.sleep(1.0 / rate_hz)

def engage_arms(steps=50, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        write_arm_sdk_pose(pose, weight=i / steps)
        time.sleep(1.0 / rate_hz)

HAND_CMD_TOPICS = {"left": "rt/dex3/left/cmd", "right": "rt/dex3/right/cmd"}
hand_pubs = {side: ChannelPublisher(topic, HandCmd_) for side, topic in HAND_CMD_TOPICS.items()}
for pub in hand_pubs.values():
    pub.Init()

import sys
sys.path.append("..")
from util import HAND_OPEN, HAND_CLOSED

def write_hand(targets, side="right", kp=0.8, kd=0.05, tau=0.02):
    msg = unitree_hg_msg_dds__HandCmd_()
    for i, q in enumerate(targets):
        cmd = msg.motor_cmd[i]
        cmd.mode = (i & 0x0F) | (1 << 4); cmd.q = float(q); cmd.dq = 0.0; cmd.tau = tau; cmd.kp = kp; cmd.kd = kd
    hand_pubs[side].Write(msg)

def open_hand(side="right"):
    return write_hand(HAND_OPEN[side], side=side)

def close_hand(side="right"):
    return write_hand(HAND_CLOSED[side], side=side)

## Task 2 - `ik_move_ee(hand, dx, dy, dz)`: position-only DLS IK step
Uses a local, URDF-derived seven-joint FK chain and numerical damped-least-squares solver. Solve a small Cartesian offset from the current end-effector pose, clip the resulting joint delta to `max_dq`, then ramp to it in speed-limited, eased steps - never jump straight to the solved pose.

In [ ]:
import numpy as np

ARM_LIMITS = {"right": [(-2.6700, 3.0890), (-2.2000, 1.5708), (-2.1817, 2.1817), (-1.0472, 2.0944), (-1.9722, 1.9722), (-1.6580, 1.6580), (-1.6580, 1.6580)], "left": [(-3.0890, 2.6700), (-1.5708, 2.2000), (-2.1817, 2.1817), (-1.0472, 2.0944), (-1.9722, 1.9722), (-1.6580, 1.6580), (-1.6580, 1.6580)]}
ARM_CHAIN = {"right": [([.003956,-.10021,.24778],[-.27931,0,0],[0,1,0]), ([0,-.038,-.013831],[.27925,0,0],[1,0,0]), ([0,-.00624,-.1032],[0,0,0],[0,0,1]), ([.015783,0,-.080518],[0,0,0],[0,1,0]), ([.100,-.001888,-.010],[0,0,0],[1,0,0]), ([.038,0,0],[0,0,0],[0,1,0]), ([.046,0,0],[0,0,0],[0,0,1])], "left": [([.003956,.10022,.24778],[.27931,0,0],[0,1,0]), ([0,.038,-.013831],[-.27925,0,0],[1,0,0]), ([0,.00624,-.1032],[0,0,0],[0,0,1]), ([.015783,0,-.080518],[0,0,0],[0,1,0]), ([.100,.001888,-.010],[0,0,0],[1,0,0]), ([.038,0,0],[0,0,0],[0,1,0]), ([.046,0,0],[0,0,0],[0,0,1])] }

def _fixed_transform(xyz, rpy):
    r, p, y = rpy; cr, sr, cp, sp, cy, sy = np.cos(r), np.sin(r), np.cos(p), np.sin(p), np.cos(y), np.sin(y)
    T = np.eye(4); T[:3,:3] = [[cy*cp, cy*sp*sr-sy*cr, cy*sp*cr+sy*sr], [sy*cp, sy*sp*sr+cy*cr, sy*sp*cr-cy*sr], [-sp, cp*sr, cp*cr]]; T[:3,3] = xyz
    return T

def _axis_rotation(axis, q):
    ax, ay, az = axis; K = np.array([[0,-az,ay],[az,0,-ax],[-ay,ax,0]], dtype=float)
    T = np.eye(4); T[:3,:3] = np.eye(3) + np.sin(q)*K + (1-np.cos(q))*(K@K); return T

def arm_fk(side, q):
    T = np.eye(4); T[:3,3] = [-.003964, 0.0, .044]
    for angle, (xyz, rpy, axis) in zip(q, ARM_CHAIN[side]):
        T = T @ _fixed_transform(xyz, rpy) @ _axis_rotation(axis, float(angle))
    palm = [.0215, -.003 if side == "right" else .003, 0.0]
    return T @ _fixed_transform(palm, [0,0,0])

def _position_jacobian(side, q, eps=1e-5):
    p0 = arm_fk(side, q)[:3,3]; J = np.zeros((3, 7))
    for i in range(7):
        q1 = q.copy(); q1[i] += eps; J[:,i] = (arm_fk(side, q1)[:3,3] - p0) / eps
    return J

def _solve_position_ik(side, q_init, target_xyz, max_iter=80, damping=.04):
    q = np.clip(q_init.copy(), *np.asarray(ARM_LIMITS[side]).T)
    for iteration in range(max_iter):
        error = target_xyz - arm_fk(side, q)[:3,3]
        if np.linalg.norm(error) < .004:
            return q, {"iterations": iteration, "error_pos_m": float(np.linalg.norm(error))}
        J = _position_jacobian(side, q); dq = J.T @ np.linalg.solve(J @ J.T + damping*damping*np.eye(3), error)
        q = np.clip(q + np.clip(dq, -.08, .08), *np.asarray(ARM_LIMITS[side]).T)
    error = target_xyz - arm_fk(side, q)[:3,3]
    return None, {"iterations": max_iter, "error_pos_m": float(np.linalg.norm(error))}

def ik_move_ee(hand, dx=0.0, dy=0.0, dz=0.0, max_speed=.25, max_dq=.12, rate_hz=50.0):
    side = "right" if str(hand).lower().startswith("r") else "left"; joints = RIGHT_ARM_JOINTS if side == "right" else LEFT_ARM_JOINTS
    current = current_upper_body_pose(); q_init = np.array([current[j] for j in joints]); target_xyz = arm_fk(side, q_init)[:3,3] + np.array([dx, dy, dz])
    q_sol, info = _solve_position_ik(side, q_init, target_xyz)
    if q_sol is None: return {"success": False, "ik": info}
    target_q = q_init + np.clip(q_sol - q_init, -max_dq, max_dq); remaining = float(np.max(np.abs(target_q-q_init)))
    rate_hz = max(1.0, float(rate_hz)); steps = max(1, int(np.ceil(remaining / max(1e-4, max_speed/rate_hz))))
    engage_arms(steps=25, rate_hz=rate_hz)
    for step in range(1, steps+1):
        ratio = step/steps; smooth = ratio*ratio*(3-2*ratio); frame = dict(current)
        frame.update({joint: float(q_init[i] + (target_q[i]-q_init[i])*smooth) for i, joint in enumerate(joints)})
        write_arm_sdk_pose(frame, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0); time.sleep(1.0/rate_hz)
    return {"success": True, "ik": info, "ee": tuple(float(x) for x in arm_fk(side, target_q)[:3,3]), "steps": steps}

In [ ]:
# ik_move_ee("right", dz=0.02)  # verify the pose and clearance before commanding

## Task 3 - FSM mode helper (for the panel below)

**FSM `500`, not `501`, for `"walk"`:** this academy's G1 units run with the waist **locked** (only `WaistYaw` is a free joint; `WaistRoll`/`WaistPitch` are absent/invalid on this hardware - see `mappings_and_constraints.html`). FSM `501` is the balanced-stand/walk id for the unlocked 3-DOF-waist variant and does not apply here, which is why `FSM_IDS["walk"]` below is `500`.

In [ ]:
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient

FSM_IDS = {"damp": 1, "prepare": 4, "walk": 500}
loco = LocoClient(); loco.SetTimeout(5.0); loco.Init()

## Task 4 - Teleoperation panel: jog buttons + FSM + release/re-engage + open/close hand
One `ipywidgets` panel: a hand selector, a 3x2 jog grid for +/-X/Y/Z (each button fires one small `ik_move_ee` step), the essential FSM buttons (`damp`/`prepare`/`walk`), `release_arms`/`engage_arms` for `rt/arm_sdk` ownership handoff, and `open_hand`/`close_hand` - the same set of controls `mode_control.py`'s Dash app exposes, native to the notebook.

In [ ]:
import ipywidgets as widgets

STEP_M = 0.02
hand_toggle = widgets.ToggleButtons(options=["right", "left"], description="hand")

def _jog(axis, sign):
    kwargs = {"dx": 0.0, "dy": 0.0, "dz": 0.0}
    kwargs["d" + axis] = sign * STEP_M
    return ik_move_ee(hand_toggle.value, **kwargs)

btn_xm = widgets.Button(description="-X"); btn_xp = widgets.Button(description="+X")
btn_ym = widgets.Button(description="-Y"); btn_yp = widgets.Button(description="+Y")
btn_zm = widgets.Button(description="-Z"); btn_zp = widgets.Button(description="+Z")
btn_xm.on_click(lambda _b: _jog("x", -1)); btn_xp.on_click(lambda _b: _jog("x", 1))
btn_ym.on_click(lambda _b: _jog("y", -1)); btn_yp.on_click(lambda _b: _jog("y", 1))
btn_zm.on_click(lambda _b: _jog("z", -1)); btn_zp.on_click(lambda _b: _jog("z", 1))
jog_pad = widgets.GridBox(
    children=[btn_xm, btn_xp, btn_ym, btn_yp, btn_zm, btn_zp],
    layout=widgets.Layout(grid_template_columns="repeat(3, 90px)"),
)

btn_damp = widgets.Button(description="Damp", button_style="danger")
btn_prepare = widgets.Button(description="Prepare")
btn_walk = widgets.Button(description="Walk")
btn_damp.on_click(lambda _b: loco.SetFsmId(FSM_IDS["damp"]))
btn_prepare.on_click(lambda _b: loco.SetFsmId(FSM_IDS["prepare"]))
btn_walk.on_click(lambda _b: loco.SetFsmId(FSM_IDS["walk"]))
mode_row = widgets.HBox([btn_damp, btn_prepare, btn_walk])

btn_release = widgets.Button(description="Release arm_sdk")
btn_engage = widgets.Button(description="Re-engage arm_sdk")
btn_release.on_click(lambda _b: release_arms())
btn_engage.on_click(lambda _b: engage_arms())
handoff_row = widgets.HBox([btn_release, btn_engage])

btn_open = widgets.Button(description="Open hand")
btn_close = widgets.Button(description="Close hand")
btn_open.on_click(lambda _b: open_hand(hand_toggle.value))
btn_close.on_click(lambda _b: close_hand(hand_toggle.value))
hand_row = widgets.HBox([btn_open, btn_close])

ui = widgets.VBox([hand_toggle, jog_pad, mode_row, handoff_row, hand_row])
display(ui)

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.

In [ ]:
open_hand("right")

In [ ]:
close_hand("right")